# exp_009 — GRPO of Qwen3-4B-Thinking-2507 (HF generation backend)

**Runtime:** Colab Pro+ — **A100 (40 or 80 GB)**.  
**Goal:** Train Qwen3-4B-Thinking-2507 with GRPO using ground-truth correctness as reward, push merged float16 weights to HuggingFace.  
**Generation backend:** vanilla HF `model.generate()` (vLLM was abandoned — too brittle on Colab).  
**Curriculum (path C — strict):** 70 prompts where the base model produces variance AND no completion ever clipped at 4096. Eliminates the ~50% wasted-step problem from the first pilot.  
**Time estimate:** Pilot (20 prompts × 4 gens) ~30 min; full strict run (70 prompts × 8 gens) ~10–12 hr/epoch on A100 — fits one Colab Pro+ session.

### Before running
1. Add `HF_TOKEN` to Colab Secrets (left sidebar → key icon).
2. Runtime → Change runtime type → **A100 GPU**, High-RAM.
3. Runtime → "Disconnect and delete runtime" (NOT just "Restart session") for a clean slate.
4. Run **Cell 1** (install). When it finishes: Runtime → Restart session.
5. Run Cell 2 onwards. Cell 4 prompts 5 file uploads:
   - `data/public.jsonl`
   - `data/splits/dev.jsonl`
   - `judger.py` (repo root)
   - `utils.py` (repo root)
   - `experiments/exp_009_grpo/sweet_spot_ids_clean.json`  ← path C strict subset (70 prompts)
6. **First run:** keep `PILOT_MODE = True`. Verify reward trends up before doing a full run.

In [1]:
# Cell 1 — Install dependencies (Aligned Stack for vLLM + GRPO)
#
# CRITICAL: Before running, do Runtime → "Disconnect and delete runtime"
# (NOT "Restart session") to ensure a clean slate.
#
# After this cell completes: Runtime → Restart session, then run from Cell 2.

# 1. Clean up potentially broken default installations
!pip uninstall -y torch torchvision torchaudio vllm bitsandbytes trl

# 2. Install PyTorch 2.5.1 built specifically for CUDA 12.4
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124

# 3. Install vLLM, correct bitsandbytes, TRL, and other utilities
!pip install vllm bitsandbytes>=0.46.1 trl==0.21.0 peft accelerate
!pip install -q sympy antlr4-python3-runtime==4.11

print("\n\n✅ Environment aligned! NOW: Go to Runtime -> Restart session, then start from Cell 2.")

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 125.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 124.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 102.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Cell 2 — Config
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

HF_REPO_ID = "TrevorDuong/qwen3-4b-thinking-grpo"  # private repo
MODEL_NAME = "Qwen/Qwen3-4B-Thinking-2507"

# ------------------------- Curriculum filtering -------------------------
# True: filter dataset to a precomputed sweet-spot ID list. False: train on all 926.
USE_CURRICULUM = True

# STRICT_CURRICULUM (PATH C):
#   True  → use sweet_spot_ids_clean.json (70 prompts; sweet-spot AND zero clipping
#           in the Phase 1 sampling). Every training step has guaranteed reward variance.
#   False → use sweet_spot_ids.json (196 prompts; includes 49 Phase 1B prompts that
#           need 8192 tokens — those will be uniformly clipped at MAX_COMPLETION_LEN=4096
#           and produce no gradient. The first pilot showed ~50% of steps wasted this way.)
STRICT_CURRICULUM = True
CURRICULUM_FILE   = "sweet_spot_ids_clean.json" if STRICT_CURRICULUM else "sweet_spot_ids.json"

# ------------------------- Pilot vs Full -------------------------
PILOT_MODE = True
PILOT_N    = 20   # ~30% of strict subset; enough to see reward trend

# ------------------------- LoRA -------------------------
MAX_SEQ_LENGTH      = 5120
MAX_PROMPT_LENGTH   = 1024
MAX_COMPLETION_LEN  = 4096
LORA_R              = 16
LORA_ALPHA          = 32

# ------------------------- GRPO -------------------------
NUM_GENERATIONS     = 8
TRAIN_BATCH_SIZE    = 1
GRAD_ACCUM_STEPS    = 4
NUM_EPOCHS          = 1
LEARNING_RATE       = 5e-6
BETA                = 0.04
TEMPERATURE         = 1.0

# ------------------------- Inference sampling (matches exp_004 baseline) -------------------------
INFER_TEMPERATURE   = 0.6
INFER_TOP_P         = 0.95
INFER_TOP_K         = 20

RANDOM_SEED         = 42

print(f"USE_CURRICULUM:    {USE_CURRICULUM}")
print(f"STRICT_CURRICULUM: {STRICT_CURRICULUM}  (file: {CURRICULUM_FILE})")
print(f"PILOT_MODE:        {PILOT_MODE}" + (f"  (PILOT_N: {PILOT_N})" if PILOT_MODE else ""))
print(f"Model:             {MODEL_NAME}")

In [2]:
# Cell 3 — Smoke load: verify Thinking-2507 loads with vanilla HF + 4-bit + LoRA.
import os
os.environ["BNB_CUDA_VERSION"] = "124"  # Force bitsandbytes to use CUDA 12.4 to avoid libnvJitLink.so.13 errors

import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME} in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,    # 0 required for GRPO — dropout during rollouts corrupts generation
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Verify chat template renders cleanly with a thinking-style turn
test_msgs = [
    {"role": "system", "content": "You are an expert mathematician."},
    {"role": "user",   "content": "What is 7 × 8?"},
]
rendered = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
print("\n=== Inference prompt (last 500 chars) ===")
print(rendered[-500:])
print("\n\nSmoke load OK. Model loaded:", MODEL_NAME)

Loading Qwen/Qwen3-4B-Thinking-2507 in 4-bit...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145

=== Inference prompt (last 500 chars) ===
<|im_start|>system
You are an expert mathematician.<|im_end|>
<|im_start|>user
What is 7 × 8?<|im_end|>
<|im_start|>assistant
<think>



Smoke load OK. Model loaded: Qwen/Qwen3-4B-Thinking-2507


In [ ]:
# Cell 4 — Upload competition files
# Need: public.jsonl (questions+answers), dev.jsonl (200 held-out), judger.py, utils.py.
# Plus the curriculum file selected by Cell 2 (STRICT_CURRICULUM flag).
import os
from google.colab import files

REQUIRED = [
    ("/content/public.jsonl",  "data/public.jsonl"),
    ("/content/dev.jsonl",     "data/splits/dev.jsonl"),
    ("/content/judger.py",     "judger.py (repo root)"),
    ("/content/utils.py",      "utils.py (repo root)"),
]
if USE_CURRICULUM:
    REQUIRED.append(
        (f"/content/{CURRICULUM_FILE}", f"experiments/exp_009_grpo/{CURRICULUM_FILE}")
    )

for path, label in REQUIRED:
    if not os.path.exists(path):
        print(f"Upload: {label}")
        files.upload()
    else:
        print(f"Already present: {path}")

import sys
sys.path.insert(0, "/content")
from judger import Judger
JUDGER = Judger()
print("Judger loaded.")

In [ ]:
# Cell 5 — Prompts (exp_004 config — best found prior to fine-tuning)
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

FEWSHOT_MATH = []

FEWSHOT_MCQ = [
    {"role": "user", "content": (
        "Find 1 over 6 + 1 over 8.\n\n"
        "Options:\nA. 7 over 24\nB. 2 over 14\nC. 1 over 4\nD. 7 over 48\n"
        "E. 2 over 24\nF. 1 over 14\nG. 1 over 2\nH. 8 over 14\nI. 0.21 Repeating\nJ. 4 over 24"
    )},
    {"role": "assistant", "content": "Common denominator is 24: 1/6 = 4/24 and 1/8 = 3/24, so 4/24 + 3/24 = 7/24. \\boxed{A}"},
    {"role": "user", "content": (
        "The function value of $\\cos(\\pi + 5i)$ is ( ).\n\n"
        "Options:\nA. -cosh5\nB. -sinh5\nC. sin5i\nD. -sin5\nE. cos5\n"
        "F. cosh5i\nG. sinh5\nH. -cos5\nI. cosh5\nJ. -sin5i"
    )},
    {"role": "assistant", "content": "$\\cos(\\pi + 5i) = -\\cos(5i) = -\\cosh(5)$, using $\\cos(\\pi+x) = -\\cos x$ and $\\cos(ix) = \\cosh x$. \\boxed{A}"},
    {"role": "user", "content": (
        "Find the range of $f(x) = \\frac{ x }{ 1+x^2 }$.\n\n"
        "Options:\nA. -\\frac{1}{3} \\le y \\le \\frac{1}{3}\nB. -\\frac{1}{\\sqrt{3}} \\le y \\le \\frac{1}{\\sqrt{3}}\n"
        "C. -\\frac{1}{4} \\le y \\le \\frac{1}{4}\nD. -\\frac{1}{\\sqrt{2}} \\le y \\le \\frac{1}{\\sqrt{2}}\n"
        "E. -\\frac{1}{\\sqrt{6}} \\le y \\le \\frac{1}{\\sqrt{6}}\nF. -\\frac{1}{2} \\le y \\le \\frac{1}{2}\n"
        "G. -\\frac{1}{\\sqrt{5}} \\le y \\le \\frac{1}{\\sqrt{5}}\nH. -1 \\le y \\le 1\n"
        "I. -\\frac{1}{\\sqrt{7}} \\le y \\le \\frac{1}{\\sqrt{7}}\nJ. -\\frac{1}{\\sqrt{4}} \\le y \\le \\frac{1}{\\sqrt{4}}"
    )},
    {"role": "assistant", "content": "$f'(x) = \\frac{1-x^2}{(1+x^2)^2} = 0$ at $x=\\pm 1$, giving $f(\\pm 1) = \\pm 1/2$. So range is $-1/2 \\le y \\le 1/2$. \\boxed{F}"},
]

In [ ]:
# Cell 6 — Build training dataset
# Filters in order:
#   1. Exclude dev.jsonl IDs (held-out test set, no leakage)
#   2. If USE_CURRICULUM: keep only IDs in CURRICULUM_FILE (strict or full sweet)
#   3. If PILOT_MODE: take first PILOT_N after shuffle
import json, random
from datasets import Dataset

random.seed(RANDOM_SEED)
LETTERS = "ABCDEFGHIJ"

# Held-out dev set — must never appear in training data.
dev_ids = set()
with open("/content/dev.jsonl") as f:
    for line in f:
        dev_ids.add(json.loads(line)["id"])
print(f"Dev IDs (excluded): {len(dev_ids)}")

# Curriculum filter — keep only prompts that produced reward variance during sampling.
if USE_CURRICULUM:
    with open(f"/content/{CURRICULUM_FILE}") as f:
        _sweet = json.load(f)
    sweet_ids = set(_sweet["sweet_ids"])
    dist = _sweet.get("distribution_num_correct") or _sweet.get("distribution", "?")
    crit = _sweet.get("selection_criteria", "")
    print(f"Curriculum: {len(sweet_ids)} IDs from {CURRICULUM_FILE}")
    print(f"  num_correct distribution: {dist}")
    if crit:
        print(f"  criteria: {crit}")
else:
    sweet_ids = None

rows = []
with open("/content/public.jsonl") as f:
    for line in f:
        ex = json.loads(line)
        if ex["id"] in dev_ids:
            continue
        if sweet_ids is not None and ex["id"] not in sweet_ids:
            continue
        # bool() wrapper required: `True and [list]` returns the list, not True.
        is_mcq = bool("options" in ex and ex["options"])
        question_text = ex["question"]
        if is_mcq:
            opts = "\n".join(f"{LETTERS[i]}. {v}" for i, v in enumerate(ex["options"]))
            question_text = f"{question_text}\n\nOptions:\n{opts}"
            sys_prompt = SYSTEM_PROMPT_MCQ
            fewshots = FEWSHOT_MCQ
        else:
            sys_prompt = SYSTEM_PROMPT_MATH
            fewshots = FEWSHOT_MATH

        msgs = [{"role": "system", "content": sys_prompt}]
        msgs.extend(fewshots)
        msgs.append({"role": "user", "content": question_text})

        rows.append({
            "prompt": msgs,
            # Serialize answer + options as JSON strings to avoid PyArrow schema-inference
            # issues on mixed list/empty-list columns.
            "answer_json":  json.dumps(ex["answer"]),
            "options_json": json.dumps(ex.get("options", [])),
            "is_mcq":       is_mcq,
            "id":           ex["id"],
        })

random.shuffle(rows)
if PILOT_MODE:
    rows = rows[:PILOT_N]
    print(f"PILOT_MODE — using {len(rows)} prompts")
else:
    print(f"Full training set — {len(rows)} prompts")

train_dataset = Dataset.from_list(rows)
print(train_dataset)
print("\nMCQ count:", sum(r["is_mcq"] for r in rows), "  Free-form:", sum(not r["is_mcq"] for r in rows))

In [ ]:
# Cell 7 — Reward functions
# Two reward signals:
#   1. correctness_reward: +1.0 if Judger.auto_judge passes, else 0.0
#   2. format_reward:     multi-component (max 0.225) to ensure gradient variance
#
# Why granular format: on hard problems all 4 completions per prompt may be wrong
# (correctness=0 for all), and binary format=0.1 fires for all → group-relative
# advantage = 0 → loss = 0 → no learning. Multi-component format rewards different
# stochastic completions differently, breaking the uniform-reward deadlock.
import re
import json as _json

def extract_post_think(text: str) -> str:
    """Return whatever follows the last </think>. If no </think>, return text."""
    idx = text.rfind("</think>")
    if idx == -1:
        return text
    return text[idx + len("</think>"):]


def correctness_reward(prompts, completions, answer_json=None, options_json=None, is_mcq=None, **kwargs):
    """GRPO reward signature: receives lists, returns list[float]."""
    rewards = []
    for i, comp in enumerate(completions):
        if isinstance(comp, list):
            text = "".join(m.get("content", "") for m in comp)
        else:
            text = str(comp)
        post = extract_post_think(text)
        gold = _json.loads(answer_json[i]) if answer_json else []
        opts = _json.loads(options_json[i]) if options_json else []
        try:
            opts_list = ([opts] * len(gold)) if gold else [None]
            ok = JUDGER.auto_judge(post, gold, opts_list)
        except Exception:
            ok = False
        rewards.append(1.0 if ok else 0.0)
    return rewards


_BOXED_RE = re.compile(r"\\boxed\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}")

def format_reward(prompts, completions, **kwargs):
    """Multi-component format reward. Components score independently so different
    stochastic completions tend to score differently → non-zero advantage even when
    all are wrong on correctness. Max possible: 0.225.
    """
    rewards = []
    for comp in completions:
        if isinstance(comp, list):
            text = "".join(m.get("content", "") for m in comp)
        else:
            text = str(comp)

        r = 0.0
        # Signal 1: model finished its thinking trace
        has_close_think = "</think>" in text
        if has_close_think:
            r += 0.05
        # Signal 2: boxed answer present anywhere
        all_boxed = _BOXED_RE.findall(text)
        if all_boxed:
            r += 0.10
        # Signal 3: boxed answer placed AFTER </think> (proper structure)
        post = extract_post_think(text)
        post_boxed = _BOXED_RE.findall(post)
        if has_close_think and post_boxed:
            r += 0.05
        # Signal 4: exactly ONE boxed in post-think (no duplicates that confuse judger)
        if len(post_boxed) == 1:
            r += 0.025

        rewards.append(r)
    return rewards


# Quick self-test
_test_text = "<think>let me reason \\boxed{42} no wait</think>\nFinal: \\boxed{42}"
_post = extract_post_think(_test_text)
assert "<think>" not in _post
assert "\\boxed{42}" in _post
_fr = format_reward([], [_test_text])
assert _fr[0] == 0.225, f"Expected 0.225, got {_fr[0]}"
# Multi-boxed should miss the "exactly one" bonus
_test_two_boxed = "<think>...</think>\nFirst: \\boxed{1} Second: \\boxed{2}"
_fr2 = format_reward([], [_test_two_boxed])
assert _fr2[0] == 0.20, f"Expected 0.20 (no exact-one bonus), got {_fr2[0]}"
print("Reward helpers self-test OK.")

In [ ]:
# Cell 8 — GRPO training (HF generation backend)
import shutil, os, gc
import sys
from unittest.mock import MagicMock

# Mock vLLM to bypass TRL's internal import bugs when vLLM is not installed
sys.modules['vllm'] = MagicMock()
sys.modules['vllm.sampling_params'] = MagicMock()
sys.modules['vllm.distributed'] = MagicMock()
sys.modules['vllm.distributed.device_communicators'] = MagicMock()
sys.modules['vllm.distributed.device_communicators.pynccl'] = MagicMock()
sys.modules['vllm.distributed.utils'] = MagicMock()
sys.modules['vllm_ascend'] = MagicMock()
sys.modules['vllm_ascend.distributed'] = MagicMock()
sys.modules['vllm_ascend.distributed.device_communicators'] = MagicMock()
sys.modules['vllm_ascend.distributed.device_communicators.pyhccl'] = MagicMock()

# Mock mergekit to bypass TRL import issues
sys.modules['mergekit'] = MagicMock()
sys.modules['mergekit.config'] = MagicMock()
sys.modules['mergekit.merge'] = MagicMock()

# Mock llm_blender
sys.modules['llm_blender'] = MagicMock()

from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback

if os.path.exists("/content/exp009_checkpoints"):
    shutil.rmtree("/content/exp009_checkpoints")

gc.collect(); torch.cuda.empty_cache()

PILOT_NUM_GENS       = 4    if PILOT_MODE else NUM_GENERATIONS
PILOT_MAX_COMPLETION = 4096 if PILOT_MODE else MAX_COMPLETION_LEN

training_args = GRPOConfig(
    output_dir="/content/exp009_checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    optim="adamw_8bit",
    bf16=True,
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=1,
    seed=RANDOM_SEED,
    report_to="none",
    # ---- GRPO-specific ----
    num_generations=PILOT_NUM_GENS,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=PILOT_MAX_COMPLETION,
    temperature=TEMPERATURE,
    beta=BETA,
    use_vllm=False,
    # NOTE: HF generation backend (no use_vllm=True). vLLM+TRL+PEFT integration on
    # Colab is brittle (stdout/fileno crashes, version churn). HF is slower but works.
)

# Monkey-patch generate to flip eval/train modes — TRL doesn't switch reliably
# under PEFT, which (1) keeps gradient-checkpointing's use_cache=False during
# generation (~5× slower), and (2) leaves any nonzero LoRA dropout active during
# rollouts (corrupts completions). lora_dropout is already 0 in Cell 3, but the
# eval/train flip is still required for the cache speedup.

# Restore original generate to prevent infinite recursion on multiple cell executions
if hasattr(type(model), "generate"):
    model.generate = type(model).generate.__get__(model)

_orig_generate = model.generate
def _eval_generate(*args, **kwargs):
    model.eval()
    out = _orig_generate(*args, **kwargs)
    model.train()
    return out
model.generate = _eval_generate

class RewardLogCallback(TrainerCallback):
    """Surface reward components per step. GRPO loss often reads ~0 even when learning
    is happening (advantages are mean-centered within a group), so the loss column
    is misleading — watch the reward components instead."""
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        keys = ("loss", "reward", "kl", "completion")
        relevant = {k: v for k, v in logs.items() if any(s in k for s in keys)}
        if relevant:
            parts = []
            for k, v in relevant.items():
                if isinstance(v, float):
                    parts.append(f"{k}={v:.4f}")
                else:
                    parts.append(f"{k}={v}")
            print(f"  [step {state.global_step}] " + "  ".join(parts))

if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[correctness_reward, format_reward],
    args=training_args,
    train_dataset=train_dataset,
    callbacks=[RewardLogCallback()],
)

print("Starting GRPO training (HF generation backend)...")
print(f"num_generations: {PILOT_NUM_GENS}  max_completion: {PILOT_MAX_COMPLETION}")
print(f"Steps/epoch: ~{len(train_dataset) // (TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)}")
print(f"Tokens/epoch: ~{len(train_dataset) * PILOT_NUM_GENS * PILOT_MAX_COMPLETION / 1e6:.1f}M\n")

trainer.train()
print("\nTraining complete. Inspect the logged reward — it should trend upward over steps.")

In [ ]:
# Cell 9 — Evaluate trained model on dev split (robust)
#
# What this fixes vs. the previous version:
#   1. Saves the LoRA adapter FIRST (insurance against eval-time crashes/hangs).
#   2. Restores the original model.generate (drops the train-mode-flip monkey-patch).
#   3. SIGALRM timeout on Judger (15 sec/q) — sympy can hang on pathological LaTeX.
#   4. Incremental write to dev_responses.jsonl — partial progress is preserved.
#   5. Per-question try/except around generate so one bad input doesn't kill the run.
#
# Optional: set DEV_LIMIT to e.g. 50 for a fast smoke check before doing all 200.
DEV_LIMIT = None  # None = all 200, or set an int to evaluate first-N only

import json, gc, signal, os
ADAPTER_PATH = "/content/exp009_adapter"
RESULTS_PATH = "/content/dev_responses.jsonl"

# 1. Save the adapter immediately as insurance ---------------------------
print(f"Saving LoRA adapter to {ADAPTER_PATH} ...")
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print("Adapter saved.\n")

# 2. Free training memory ------------------------------------------------
try:
    del trainer
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

# 3. Restore original generate (override Cell 8's eval/train flip wrapper) -
#    During eval we don't want the wrapper toggling train mode after each call.
model.generate = type(model).generate.__get__(model)
model.eval()

# 4. SIGALRM-based Judger timeout (sympy hang protection) ----------------
class JudgerTimeout(Exception):
    pass
def _alarm(signum, frame):
    raise JudgerTimeout()
signal.signal(signal.SIGALRM, _alarm)

def safe_judge(post_text, gold, options):
    signal.alarm(15)
    try:
        opts_list = ([options] * len(gold)) if gold else [None]
        return JUDGER.auto_judge(post_text, gold, opts_list)
    except JudgerTimeout:
        return False
    except Exception:
        return False
    finally:
        signal.alarm(0)

# 5. Load dev split ------------------------------------------------------
dev_rows = []
with open("/content/dev.jsonl") as f:
    for line in f:
        dev_rows.append(json.loads(line))
if DEV_LIMIT is not None:
    dev_rows = dev_rows[:DEV_LIMIT]
print(f"Dev questions to evaluate: {len(dev_rows)}")

# 6. Eval loop with incremental save ------------------------------------
correct = 0; mcq_c = 0; mcq_n = 0; ff_c = 0; ff_n = 0
out_f = open(RESULTS_PATH, "w")

for i, ex in enumerate(dev_rows):
    is_mcq = bool("options" in ex and ex["options"])
    qt = ex["question"]
    if is_mcq:
        opts = "\n".join(f"{LETTERS[j]}. {v}" for j, v in enumerate(ex["options"]))
        qt = f"{qt}\n\nOptions:\n{opts}"
        msgs = [{"role": "system", "content": SYSTEM_PROMPT_MCQ}] + FEWSHOT_MCQ + [{"role": "user", "content": qt}]
    else:
        msgs = [{"role": "system", "content": SYSTEM_PROMPT_MATH}] + FEWSHOT_MATH + [{"role": "user", "content": qt}]

    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    attn = torch.ones_like(inputs)

    text = ""
    try:
        with torch.no_grad():
            gen = model.generate(
                input_ids=inputs,
                attention_mask=attn,
                max_new_tokens=4096,
                do_sample=True,
                temperature=INFER_TEMPERATURE,
                top_p=INFER_TOP_P,
                top_k=INFER_TOP_K,
                pad_token_id=tokenizer.pad_token_id,
            )
        text = tokenizer.decode(gen[0][inputs.shape[1]:], skip_special_tokens=True)
    except Exception as e:
        print(f"  ⚠ Q{i} (id={ex['id']}) generate failed: {type(e).__name__}: {e}")

    post = extract_post_think(text)
    gold = ex["answer"]
    opts = ex.get("options", [])
    ok = safe_judge(post, gold, opts)

    if ok:
        correct += 1
        if is_mcq: mcq_c += 1
        else:      ff_c  += 1
    if is_mcq: mcq_n += 1
    else:      ff_n  += 1

    rec = {"id": ex["id"], "response": text, "correct": ok, "is_mcq": is_mcq}
    out_f.write(json.dumps(rec) + "\n"); out_f.flush()

    if (i + 1) % 10 == 0:
        running = 100 * correct / (i + 1)
        print(f"  [{i+1}/{len(dev_rows)}] running acc: {running:.1f}%  (MCQ {mcq_c}/{mcq_n}, FF {ff_c}/{ff_n})")

out_f.close()

print("\n=== Dev results ===")
print(f"Overall:    {correct}/{len(dev_rows)}  ({100*correct/len(dev_rows):.2f}%)")
if mcq_n: print(f"MCQ:        {mcq_c}/{mcq_n}  ({100*mcq_c/mcq_n:.2f}%)")
if ff_n:  print(f"Free-form:  {ff_c}/{ff_n}  ({100*ff_c/ff_n:.2f}%)")
print("Baseline (exp_004): 55.33% / MCQ 63.20% / FF 51.40%")
print(f"\nSaved {RESULTS_PATH} — download via Files panel.")

In [ ]:
# Cell 10 — Merge LoRA adapter into fp16 base, save merged model
# Self-contained: loads adapter from /content/exp009_adapter (saved at start of Cell 9).
# Frees the in-memory training model first to make room for the fp16 base (~16 GB).
#
# Run this only after Cell 9's dev results look good — merging + uploading takes ~30 min.
import os, gc, torch
from peft import PeftModel

ADAPTER_PATH = "/content/exp009_adapter"
MERGED_PATH  = "/content/exp009_merged"

# If Cell 9 wasn't run, save the adapter now from the in-memory model.
if not os.path.exists(ADAPTER_PATH):
    print(f"No adapter on disk — saving from in-memory model to {ADAPTER_PATH} ...")
    model.save_pretrained(ADAPTER_PATH)
    tokenizer.save_pretrained(ADAPTER_PATH)
else:
    print(f"Using adapter on disk: {ADAPTER_PATH}")

# Free GPU memory before loading fp16 base (~16 GB)
for var in ("trainer", "model", "peft_model", "merged", "base"):
    if var in dir():
        try: exec(f"del {var}")
        except NameError: pass
gc.collect(); torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()
print(f"GPU free before load: {free/1e9:.1f} GB / {total/1e9:.1f} GB")

print(f"\nLoading {MODEL_NAME} in float16 ...")
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("Loading adapter and merging ...")
peft_model = PeftModel.from_pretrained(base, ADAPTER_PATH)
merged = peft_model.merge_and_unload()
merged.save_pretrained(MERGED_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_PATH)

print(f"\nMerged model saved to {MERGED_PATH}")
!du -sh {MERGED_PATH}

In [ ]:
# Cell 11 — Push to HuggingFace (private repo)
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO_ID, private=True, exist_ok=True)
print(f"Repo: https://huggingface.co/{HF_REPO_ID}")

print("Uploading merged model (~8GB)...")
api.upload_folder(
    folder_path=MERGED_PATH,
    repo_id=HF_REPO_ID,
    repo_type="model",
    token=HF_TOKEN,
)
print(f"Done. Set HF_REPO_ID = '{HF_REPO_ID}' in the Kaggle inference notebook.")